In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,GridSearchCV,cross_val_score
from sklearn.tree import DecisionTreeRegressor,DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier,GradientBoostingClassifier,RandomForestClassifier
from sklearn.svm import SVR,SVC
from sklearn.metrics import accuracy_score,r2_score,confusion_matrix,precision_score
from sklearn.linear_model import LinearRegression,LogisticRegression
import xgboost as xgb
from sklearn.naive_bayes import MultinomialNB,GaussianNB,BernoulliNB
import seaborn as sns

In [ ]:
df=pd.read_csv('/kaggle/input/datasets/organizations/uciml/sms-spam-collection-dataset/spam.csv'
,encoding='latin-1')
df.head()

In [ ]:
df.info()
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'],inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['v1']=le.fit_transform(df['v1'])

In [ ]:
df['v1'].value_counts()

In [ ]:
!pip install nltk
import nltk
nltk.download('punkt_tab')


In [ ]:
df['char_length']=df['v2'].apply(len)
df['word_count']=df['v2'].apply(lambda x:len(nltk.word_tokenize(x)))
df['sentence_count']=df['v2'].apply(lambda x:len(nltk.sent_tokenize(x)))

In [ ]:
sns.displot(df['char_length'],kind='kde')

In [ ]:
sns.displot(df['word_count'],kind='kde')

In [ ]:
sns.displot(df['sentence_count'],kind='kde')

In [ ]:
df[df['v1']==1].describe()

In [ ]:
new_df=df.drop(columns=['v2'])
sns.heatmap(new_df.corr(),annot=True)

In [ ]:
df.drop(columns=['word_count','sentence_count'],inplace=True)

In [ ]:
df.head()

****Text Preprocessing using nltk****

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords
import string

In [ ]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [ ]:
def transform_text(text):
    text=text.lower()
    text=nltk.word_tokenize(text)
    y=[]
    for i in text:
      if i.isalnum():
        y.append(i)
    text=y[:]
    y.clear()
    for i in text:
      if i not in stopwords.words('english') and i not in string.punctuation:
        y.append(i)
    text=y[:]
    y.clear()
    for i in text:
      y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
df['transformed_text']=df['v2'].apply(transform_text)

In [ ]:
df.rename(columns={'v1':'spam','v2':'text'},inplace=True)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

In [ ]:
tf=TfidfVectorizer()
cv=CountVectorizer()

In [ ]:
X_tf=tf.fit_transform(df['transformed_text']).toarray()
X_cv=cv.fit_transform(df['transformed_text']).toarray()

In [ ]:
y=df['spam']

In [ ]:
mnb=MultinomialNB()
gnb=GaussianNB()
bnb=BernoulliNB()
xgbc=xgb.XGBClassifier()
adbc=AdaBoostClassifier()
svc=SVC()
rf=RandomForestClassifier()


In [ ]:
models={'mnb':mnb,'gnb':gnb,'bnb':bnb,'xgbc':xgbc,'adbc':adbc,'svc':svc,'rf':rf}

In [ ]:
scores_tf=[]
model_names=[]
scores_cv=[]
precision_cv=[]
precision_tf=[]
for model in models.keys():
    cf=models[model]
    scores_tf.append(cross_val_score(cf,X_tf,y,scoring='accuracy',cv=3).mean())
    precision_tf.append(cross_val_score(cf,X_tf,y,scoring='precision',cv=3).mean())
    precision_cv.append(cross_val_score(cf,X_cv,y,scoring='precision',cv=3).mean())
    scores_cv.append(cross_val_score(cf,X_cv,y,scoring='accuracy',cv=3).mean())
    model_names.append(model)
    
    
    

In [ ]:
all_scores=pd.DataFrame({'model_name':model_names,'tfid_scores':scores_tf,'precision_tf':precision_tf,'cv_scores':scores_cv,'precision_cv':precision_cv}).sort_values(by=['precision_tf','precision_cv'],ascending=False)

In [ ]:
all_scores

In [ ]:
from sklearn.ensemble import VotingClassifier
estimators=[('mnb',mnb),('rf',rf),('svc',svc)]
vc=VotingClassifier(estimators=estimators)
vc_score=np.mean(cross_val_score(vc,X_tf,y,scoring='accuracy',cv=5))
vc_precision=np.mean(cross_val_score(vc,X_tf,y,scoring='precision',cv=5))

In [ ]:
print(all_scores.loc[all_scores['model_name']=='mnb','tfid_scores'].values)
print(all_scores.loc[all_scores['model_name']=='mnb','precision_tf'].values)
print(vc_score)
print(vc_precision)

**This means our choice should be VotingEnsemble with (mnb,rf,svc) and tfId_vectorizer showing Precision of 1.0 and accuracy of 0.9684**